In [1]:
# =========================
# PATHS
# =========================

train_path = "EuroSAT_Dataset\\EuroSAT\\train"
val_path   = "EuroSAT_Dataset\\EuroSAT\\val"
test_path  = "EuroSAT_Dataset\\EuroSAT_test_flat"

# =========================
# HYPERPARAMETERS
# =========================

NUM_CLASSES = 10

BATCH_SIZE = 64
EPOCHS = 30

LR = 1e-4
WEIGHT_DECAY = 1e-4

IMAGE_SIZE = 224

NUM_WORKERS = 0

In [2]:
import os
import cv2
import torch
import timm
import numpy as np

import torch.nn as nn

from tqdm.auto import tqdm

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

In [3]:
class EuroSATRGBDataset(Dataset):

    def __init__(self, root_dir):

        self.samples = []

        classes = sorted([
            d for d in os.listdir(root_dir)
            if os.path.isdir(
                os.path.join(root_dir, d)
            )
        ])

        self.class_to_idx = {
            cls: idx
            for idx, cls in enumerate(classes)
        }

        print("Classes:", classes)

        for cls in classes:

            cls_path = os.path.join(
                root_dir,
                cls
            )

            count = 0

            for file in os.listdir(cls_path):

                if file.lower().endswith(
                    (".jpg", ".jpeg", ".png")
                ):

                    self.samples.append(
                        (
                            os.path.join(
                                cls_path,
                                file
                            ),
                            self.class_to_idx[cls]
                        )
                    )

                    count += 1

            print(
                f"{cls}: {count}"
            )

        print(
            "Total Samples:",
            len(self.samples)
        )

    def __len__(self):

        return len(self.samples)

    def __getitem__(self, idx):

        path, label = self.samples[idx]

        img = cv2.imread(path)

        img = cv2.cvtColor(
            img,
            cv2.COLOR_BGR2RGB
        )

        img = cv2.resize(
            img,
            (IMAGE_SIZE, IMAGE_SIZE)
        )

        img = torch.tensor(
            img,
            dtype=torch.float32
        )

        img = img / 255.0

        img = img.permute(
            2,
            0,
            1
        )

        return img, label
    

In [4]:
train_dataset = EuroSATRGBDataset(
    train_path
)

val_dataset = EuroSATRGBDataset(
    val_path
)

print()

print(
    "Train Samples:",
    len(train_dataset)
)

print(
    "Val Samples:",
    len(val_dataset)
)

Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
AnnualCrop: 2100
Forest: 2100
HerbaceousVegetation: 2100
Highway: 1750
Industrial: 1750
Pasture: 1400
PermanentCrop: 1750
Residential: 2100
River: 1750
SeaLake: 2100
Total Samples: 18900
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
AnnualCrop: 450
Forest: 450
HerbaceousVegetation: 450
Highway: 375
Industrial: 375
Pasture: 300
PermanentCrop: 375
Residential: 450
River: 375
SeaLake: 450
Total Samples: 4050

Train Samples: 18900
Val Samples: 4050


In [5]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

In [6]:
images, labels = next(
    iter(train_loader)
)

print(images.shape)
print(labels.shape)

torch.Size([64, 3, 224, 224])
torch.Size([64])


In [7]:
model = timm.create_model(
    "convnext_tiny",
    pretrained=True,
    num_classes=NUM_CLASSES,
    in_chans=3
)

In [8]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():

    print(
        torch.cuda.get_device_name(0)
    )

model = model.to(device)

Device: cuda
NVIDIA GeForce RTX 4050 Laptop GPU


In [9]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

scaler = torch.amp.GradScaler()

In [10]:
def train_one_epoch():

    model.train()

    running_loss = 0

    correct = 0
    total = 0

    pbar = tqdm(
        train_loader,
        desc="Training",
        leave=True
    )

    for images, labels in pbar:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

        scaler.scale(
            loss
        ).backward()

        scaler.step(
            optimizer
        )

        scaler.update()

        running_loss += loss.item()

        preds = outputs.argmax(1)

        correct += (
            preds == labels
        ).sum().item()

        total += labels.size(0)

        pbar.set_postfix({
            "loss":
            f"{loss.item():.4f}",

            "acc":
            f"{correct/total:.4f}"
        })

    return (
        running_loss / len(train_loader),
        correct / total
    )

In [11]:
@torch.no_grad()
def validate():

    model.eval()

    running_loss = 0

    correct = 0
    total = 0

    pbar = tqdm(
        val_loader,
        desc="Validation",
        leave=True
    )

    for images, labels in pbar:

        images = images.to(device)

        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        running_loss += loss.item()

        preds = outputs.argmax(1)

        correct += (
            preds == labels
        ).sum().item()

        total += labels.size(0)

        pbar.set_postfix({
            "acc":
            f"{correct/total:.4f}"
        })

    return (
        running_loss / len(val_loader),
        correct / total
    )

In [13]:
best_acc = 0.0

for epoch in range(EPOCHS):

    print(
        f"\n========== "
        f"Epoch {epoch+1}/{EPOCHS} "
        f"=========="
    )

    train_loss, train_acc = (
        train_one_epoch()
    )

    val_loss, val_acc = (
        validate()
    )

    scheduler.step()

    print(
        f"\nTrain Loss: "
        f"{train_loss:.4f}"
    )

    print(
        f"Train Acc : "
        f"{train_acc:.4f}"
    )

    print(
        f"Val Loss  : "
        f"{val_loss:.4f}"
    )

    print(
        f"Val Acc   : "
        f"{val_acc:.4f}"
    )

    if val_acc > best_acc:

        best_acc = val_acc

        torch.save(
            model.state_dict(),
            "best_convnext_rgb.pth"
        )

        print(
            f"Saved Best Model "
            f"(Acc={best_acc:.4f})"
        )

print()

print(
    "Best Validation Accuracy:",
    best_acc
)


========== Epoch 1/30 ==========


Training:   0%|          | 0/296 [00:00<?, ?it/s]

Validation:   0%|          | 0/64 [00:00<?, ?it/s]


Train Loss: 0.4296
Train Acc : 0.8482
Val Loss  : 0.2256
Val Acc   : 0.9235
Saved Best Model (Acc=0.9235)

========== Epoch 2/30 ==========


Training:   0%|          | 0/296 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/64 [00:00<?, ?it/s]


Train Loss: 0.0000
Train Acc : 1.0000
Val Loss  : 0.1105
Val Acc   : 0.9800
Saved Best Model (Acc=0.9800)

========== Epoch 30/30 ==========


Training:   0%|          | 0/296 [00:00<?, ?it/s]

Validation:   0%|          | 0/64 [00:00<?, ?it/s]


Train Loss: 0.0000
Train Acc : 1.0000
Val Loss  : 0.1105
Val Acc   : 0.9800

Best Validation Accuracy: 0.98


In [14]:
model.load_state_dict(
    torch.load(
        "best_convnext_rgb.pth"
    )
)

model.eval()

ConvNeXt(
  (stem): Sequential(
    (0): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
    (1): LayerNorm2d((96,), eps=1e-06, elementwise_affine=True, bias=True)
  )
  (stages): Sequential(
    (0): ConvNeXtStage(
      (downsample): Identity()
      (blocks): Sequential(
        (0): ConvNeXtBlock(
          (conv_dw): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (norm): LayerNorm((96,), eps=1e-06, elementwise_affine=True, bias=True)
          (mlp): Mlp(
            (fc1): Linear(in_features=96, out_features=384, bias=True)
            (act): GELU()
            (drop1): Dropout(p=0.0, inplace=False)
            (norm): Identity()
            (fc2): Linear(in_features=384, out_features=96, bias=True)
            (drop2): Dropout(p=0.0, inplace=False)
          )
          (shortcut): Identity()
          (drop_path): Identity()
        )
        (1): ConvNeXtBlock(
          (conv_dw): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), paddi